# AIC Screener — fine-tuning a small model with Unsloth

All In Capital pre-seed screener · lecture 24 Nov (feedback loop → fine-tuning).

**What this does:** trains a small open model (Qwen3-4B-Instruct, LoRA, 4-bit) on the screener's feedback — score cards
with the partners' corrections, and email drafts with Riya's edits — then checks it on a held-out test split and says
whether it clears the bar.

**What it does not do:** decide anything, or deploy anything. A tuned model that does not clear the bar on the test split
is **not** used. Fine-tuning does not replace the feedback loop; the loop keeps collecting data for the next retrain.

**Before you run it**
1. Check readiness in the repo: `python -m aic.learning readiness` (or the dashboard's *Feedback & fine-tuning* page).
   If it says *Not yet*, a real run will not produce a useful model. You can still run with `SMOKE_TEST = True`
   to prove the pipeline works end to end.
2. Export the data. Two formats:
   - **Per criterion (recommended on a free T4):** `python -m aic.learning export --per-criterion` writes
     `data/finetune/per_criterion/`. One rubric criterion per example, with only the inputs that criterion needs
     (a fixed map in `config/rubric.yaml → finetune.criterion_inputs`). The longest example is about 7k tokens.
   - **Full score card:** `python -m aic.learning export` writes `data/finetune/`. It uses the same prompt as the live app
     but is about 30–40k tokens per example, so it needs a large GPU.
3. Colab: **Runtime → Change runtime type → T4 GPU** (free tier) or better.

**Privacy.** The files hold founder data. Colab runs on Google's machines, so uploading them sends that data to Google.
The lecture's self-managed route means: run this same notebook on the fund's own GPU (Jupyter) if the data must stay in-house.

**Metrics are defined before training** (lecture): baseline = the untuned model on the test split; target = the bar in
`manifest.json` (a PROPOSAL for Kushal, from `config/rubric.yaml → finetune.bar`).

## 1. Install (Unsloth's current Colab install, with its pinned versions)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # local / own GPU
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 2. Settings

- `SMOKE_TEST = True` trains on a few **made-up** examples generated below, only to prove the pipeline runs.
  Its results say nothing about quality and must never be reported as such.
- `MAX_SEQ_LENGTH = None` uses the manifest's suggestion (8,192 for per-criterion data). Examples longer than this are
  **excluded and counted**, never truncated, because truncation would cut off the answer the model is meant to learn.
  Step 4 reports exactly what was kept.

In [ ]:
SMOKE_TEST = False            # True: made-up data, pipeline check only
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"   # small instruct model (lecture: 2-8B Gemma/Qwen/Llama)
CHAT_TEMPLATE = "qwen3-instruct"
MAX_SEQ_LENGTH = None         # None: the manifest's suggestion (8192 per-criterion; full cards need a large GPU)
EPOCHS = 3                    # upper bound: training stops early when validation loss rises
LEARNING_RATE = 2e-4
LORA_RANK = 16
SEED = 3407
DATA_DIR = "/content/finetune"

## 3. Load the exported data

Upload `train.jsonl`, `val.jsonl`, `test.jsonl` and `manifest.json` from `data/finetune/per_criterion/` (or `data/finetune/` for full cards), or put them in `DATA_DIR`.

In [ ]:
import json, os, random
from pathlib import Path

os.makedirs(DATA_DIR, exist_ok=True)
need = ["train.jsonl", "val.jsonl", "test.jsonl", "manifest.json"]
if not SMOKE_TEST and not all((Path(DATA_DIR) / f).exists() for f in need):
    try:
        from google.colab import files
        print("Upload", need)
        for name, data in files.upload().items():
            (Path(DATA_DIR) / name).write_bytes(data)
    except ImportError:
        pass

def read_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []

def task_of(row):
    sysmsg = row["messages"][0]["content"]
    return "email" if sysmsg.startswith("Rewrite this All In Capital outreach draft") else "scoring"

if SMOKE_TEST:
    # MADE-UP examples, clearly not real feedback: only to check that training and evaluation run.
    random.seed(SEED)
    def fake(i):   # alternates the two export formats so both evaluation paths run
        n = random.randint(0, 5)
        ans = {"score": n, "rationale": f"{n * 4} paying customers"}
        full = i % 2 == 0
        return {"messages": [
            {"role": "system", "content": ("Score the whole card. " if full else "Score ONE criterion: T1. ") + "Return JSON only."},
            {"role": "user", "content": json.dumps({"company": f"SmokeCo{i}", "paying_customers": n * 4, "pilots": 5 - n})},
            {"role": "assistant", "content": json.dumps({"criteria": {"T1": ans}} if full else {"criterion": "T1", **ans})}]}
    rows = [fake(i) for i in range(60)]
    train, val, test = rows[:45], rows[45:54], rows[54:]
    manifest = {"bar": {"disagreement_above": 0.30}, "bar_is_proposal": True, "pairs": 60, "granularity": "smoke_test",
                "suggested_max_seq_length": 2048, "note": "SMOKE TEST: made-up data"}
else:
    train, val, test = (read_jsonl(Path(DATA_DIR) / f"{s}.jsonl") for s in ("train", "val", "test"))
    manifest = json.loads((Path(DATA_DIR) / "manifest.json").read_text())

BAR = manifest["bar"]["disagreement_above"]
if MAX_SEQ_LENGTH is None:
    MAX_SEQ_LENGTH = manifest.get("suggested_max_seq_length", 8192)
BATCH, ACCUM = (1, 8) if MAX_SEQ_LENGTH > 4096 else (2, 4)    # same effective batch; long sequences need less memory per step
print(f"format: {manifest.get('granularity', 'unknown')} · MAX_SEQ_LENGTH {MAX_SEQ_LENGTH}")
print(f"train {len(train)} · val {len(val)} · test {len(test)} · bar: disagreement below {BAR:.0%} "
      f"({'PROPOSAL for Kushal' if manifest.get('bar_is_proposal') else 'agreed'})")
if not train:
    raise SystemExit("No training pairs. The feedback loop has not produced labelled data yet "
                     "(see `python -m aic.learning readiness`). Set SMOKE_TEST = True to test the pipeline only.")
if not test:
    print("WARNING: no held-out test pairs, so the tuned model cannot be judged. Do not deploy it.")

## 4. Load the base model, apply the chat template, and drop examples that do not fit

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template = CHAT_TEMPLATE)

def n_tokens(messages):
    return len(tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = False))

def fit(rows, name):
    kept, dropped = [], {}
    for r in rows:
        if n_tokens(r["messages"]) <= MAX_SEQ_LENGTH:
            kept.append(r)
        else:
            dropped[task_of(r)] = dropped.get(task_of(r), 0) + 1
    print(f"{name}: kept {len(kept)} of {len(rows)}" + (f"; too long (excluded, not truncated): {dropped}" if dropped else ""))
    return kept

train, val, test = fit(train, "train"), fit(val, "val"), fit(test, "test")
if not train:
    raise SystemExit(f"Nothing fits in MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. Use a bigger GPU and raise it, "
                     "or export shorter (per-criterion) scoring pairs.")

## 5. Baseline: the untuned model on the test split (metrics defined before training)

- **Scoring pairs:** per criterion, does the model's 0–5 score equal the label (the partner-corrected score)?
  Disagreement = share of criteria that differ. Also: within ±1, mean absolute error, and how often the output is valid JSON.
- **Email pairs:** how far the model's email is from what Riya approved (0 = identical, 1 = rewritten), the same measure as the dashboard.

In [ ]:
from difflib import SequenceMatcher

def generate(messages, max_new_tokens):
    text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def parse_json(s):
    a, b = s.find("{"), s.rfind("}")
    try:
        return json.loads(s[a:b + 1]) if a >= 0 and b > a else None
    except json.JSONDecodeError:
        return None

def evaluate(rows, label):
    FastLanguageModel.for_inference(model)
    crit_total = crit_diff = crit_within1 = abs_err = valid = n_scoring = 0
    email_change = []
    for r in rows:
        want = r["messages"][2]["content"]
        if task_of(r) == "scoring":
            n_scoring += 1
            want_j = json.loads(want)
            per_crit = "criteria" not in want_j                 # one criterion per pair, or the whole card
            want_c = {want_j["criterion"]: want_j} if per_crit else want_j["criteria"]
            got = parse_json(generate(r["messages"][:2], 768 if per_crit else 2048))
            if not isinstance(got, dict):
                got_c = {}
            elif "criteria" in got:
                got_c = got["criteria"] if isinstance(got["criteria"], dict) else {}
            else:
                got_c = {got.get("criterion", next(iter(want_c))): got}
            valid += got is not None
            for k, w in want_c.items():
                g = got_c.get(k, {}).get("score") if isinstance(got_c.get(k), dict) else None
                crit_total += 1
                if not isinstance(g, (int, float)):
                    crit_diff += 1          # missing or unparseable counts as a disagreement, never skipped
                    abs_err += 5
                    continue
                crit_diff += int(g != w["score"])
                crit_within1 += int(abs(g - w["score"]) <= 1)
                abs_err += abs(g - w["score"])
        else:
            want_j = json.loads(want)
            got = parse_json(generate(r["messages"][:2], 768)) or {}
            a = want_j["subject"] + "\n" + want_j["body"]
            b = str(got.get("subject", "")) + "\n" + str(got.get("body", ""))
            email_change.append(1 - SequenceMatcher(None, a, b).ratio())
    res = {"label": label, "test_pairs": len(rows),
           "scoring": {"pairs": n_scoring, "criteria": crit_total,
                       "disagreement": round(crit_diff / crit_total, 3) if crit_total else None,
                       "within_1": round(crit_within1 / crit_total, 3) if crit_total else None,
                       "mean_abs_error": round(abs_err / crit_total, 2) if crit_total else None,
                       "valid_json": round(valid / n_scoring, 3) if n_scoring else None},
           "email": {"pairs": len(email_change),
                     "mean_change_vs_riya": round(sum(email_change) / len(email_change), 3) if email_change else None}}
    print(json.dumps(res, indent = 1))
    return res

baseline = evaluate(test, "baseline (untuned)") if test else None

## 6. Add LoRA adapters and train, stopping when validation loss starts rising

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from datasets import Dataset
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

def to_text(rows):
    return Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False,
                                                                     add_generation_prompt = False)} for r in rows])

train_ds, val_ds = to_text(train), (to_text(val) if val else None)
use_eval = val_ds is not None and len(val_ds) > 0

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds if use_eval else None,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 1)] if use_eval else [],
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = ACCUM,
        warmup_steps = 5,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = "outputs",
        eval_strategy = "epoch" if use_eval else "no",
        save_strategy = "epoch" if use_eval else "no",
        load_best_model_at_end = use_eval,
        metric_for_best_model = "eval_loss" if use_eval else None,
        greater_is_better = False if use_eval else None,
        report_to = "none",
    ),
)
# Learn only the answers (score card / Riya's email), not the prompts.
trainer = train_on_responses_only(trainer)
if not use_eval:
    print("WARNING: no validation split, so there is no early stopping. Watch for overfitting.")
stats = trainer.train()

## 7. Loss curves

Training loss should fall. Validation loss falls, then flattens or rises; the run keeps the epoch with the lowest validation loss.

In [ ]:
import matplotlib.pyplot as plt
h = trainer.state.log_history
tr = [(x["step"], x["loss"]) for x in h if "loss" in x]
ev = [(x["step"], x["eval_loss"]) for x in h if "eval_loss" in x]
plt.figure(figsize = (7, 3.5))
if tr: plt.plot(*zip(*tr), label = "training loss", color = "#2a78d6", linewidth = 2)
if ev: plt.plot(*zip(*ev), "o-", label = "validation loss", color = "#eb6834", linewidth = 2)
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha = .3); plt.tight_layout(); plt.show()
if ev:
    print("best validation loss", min(v for _, v in ev))

## 8. Evaluate the tuned model on the held-out test split, and decide

In [ ]:
tuned = evaluate(test, "fine-tuned") if test else None

def verdict(base, tuned):
    if SMOKE_TEST:
        return "SMOKE TEST: the pipeline ran end to end. These numbers come from made-up data and say nothing about quality."
    if not tuned:
        return "No test split: the tuned model cannot be judged. Do not deploy it."
    d, b = tuned["scoring"]["disagreement"], (base or {}).get("scoring", {}).get("disagreement")
    e, eb = tuned["email"]["mean_change_vs_riya"], (base or {}).get("email", {}).get("mean_change_vs_riya")
    lines = []
    if d is not None:
        ok = d < BAR and (b is None or d < b)
        lines.append(f"Scoring: disagreement {d:.0%} (baseline {b if b is None else f'{b:.0%}'}; bar below {BAR:.0%}) -> "
                     + ("clears the bar" if ok else "does NOT clear the bar"))
    if e is not None:
        lines.append(f"Email: change vs Riya {e:.0%} (baseline {eb if eb is None else f'{eb:.0%}'}); lower is better")
    if d is not None and not (d < BAR and (b is None or d < b)):
        lines.append("Decision: do not deploy. Keep the feedback loop running and collect more data (lecture: it is acceptable not to deploy).")
    else:
        lines.append("Decision for a person: candidate only. Kushal confirms the bar; then serve it behind its own GPU endpoint and keep the feedback loop on.")
    return "\n".join(lines)

summary = {"base_model": BASE_MODEL, "smoke_test": SMOKE_TEST, "max_seq_length": MAX_SEQ_LENGTH,
           "train": len(train), "val": len(val), "test": len(test), "bar": manifest["bar"],
           "bar_is_proposal": manifest.get("bar_is_proposal", True), "baseline": baseline, "tuned": tuned,
           "best_val_loss": min((x["eval_loss"] for x in trainer.state.log_history if "eval_loss" in x), default = None),
           "verdict": verdict(baseline, tuned)}
print(summary["verdict"])
Path("finetune_result.json").write_text(json.dumps(summary, indent = 1))

## 9. Save the adapter (and download the result)

The LoRA adapter is small and sits on top of the base model. Keep it with `finetune_result.json`, which records the data
sizes, the bar, the baseline and the tuned scores.

**Serving** (lecture): a fine-tuned model needs its own GPU-backed endpoint (about $300–600/month; serverless GPU
is cheaper at low volume but has a 2–3 minute cold start). It cannot run on Vercel. Merged or GGUF exports (below, off
by default) are for vLLM or Ollama.

In [ ]:
model.save_pretrained("aic_lora")
tokenizer.save_pretrained("aic_lora")
!zip -qr aic_lora.zip aic_lora finetune_result.json
try:
    from google.colab import files
    files.download("finetune_result.json")
    files.download("aic_lora.zip")
except ImportError:
    print("Saved aic_lora/ and finetune_result.json")

if False:  # merged 16-bit weights for vLLM
    model.save_pretrained_merged("aic_merged_16bit", tokenizer, save_method = "merged_16bit")
if False:  # GGUF for Ollama / llama.cpp
    model.save_pretrained_gguf("aic_gguf", tokenizer, quantization_method = "q4_k_m")